In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

cwd = os.getcwd()
root = os.path.dirname(cwd) if cwd.endswith('/code') else cwd

df = pd.read_csv(root + '/artifacts/results_few_models.csv')

traj = (
    df.groupby(['model_id', 'train_tokens_real'], as_index=False)
      .agg(
          val_loss=('val_loss', 'mean'),
          val_loss_std=('val_loss', 'std'),
          mu=('mu_landscape', 'mean'),
          mu_std=('mu_landscape', 'std'),
      )
)
traj['mu'] = np.clip(traj['mu'].to_numpy(), 1e-12, None)
traj['val_loss_std'] = traj['val_loss_std'].fillna(0.0)
traj['mu_std'] = traj['mu_std'].fillna(0.0)
model_ids = sorted(traj['model_id'].unique())

print('rows:', len(df), 'models:', model_ids)

In [ ]:
palette = plt.get_cmap('tab10').colors

fig, ax = plt.subplots(figsize=(10.8, 6.0))
for i, model_id in enumerate(model_ids):
    g = traj[traj['model_id'] == model_id].sort_values('train_tokens_real')
    color = palette[i % len(palette)]
    ax.plot(g['train_tokens_real'], g['val_loss'], '-', lw=2.0, color=color, label=f'{model_id} loss')
    ax.errorbar(g['train_tokens_real'], g['val_loss'], yerr=g['val_loss_std'], fmt='none', ecolor=color, alpha=0.28, elinewidth=0.8, capsize=0)

ax2 = ax.twinx()
for i, model_id in enumerate(model_ids):
    g = traj[traj['model_id'] == model_id].sort_values('train_tokens_real')
    color = palette[i % len(palette)]
    ax2.plot(g['train_tokens_real'], g['mu'], '--', lw=1.7, color=color, alpha=0.78)
    ax2.errorbar(g['train_tokens_real'], g['mu'], yerr=g['mu_std'], fmt='none', ecolor=color, alpha=0.22, elinewidth=0.7, capsize=0)

ax.set_xscale('log')
ax.set_xlabel('Training tokens (log scale)')
ax.set_ylabel('Validation loss')
ax2.set_ylabel('μ')
ax.grid(True, alpha=0.22)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, frameon=False, fontsize=9, loc='upper right')
fig.tight_layout()

out_path = root + '/paper/figures/fig_few_models_dynamics.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print('saved:', out_path)

plt.show()

In [ ]:
print('Run order: 0 -> 1')